In [1]:
import json
import numpy as np
import pandas as pd
from llms.genserv.client import GenerationServiceClient
from evalserv_client import EvaluationServiceClient
from utils_rollout import generate_responses
from tasks import get_task
from utils import print_colored

model_name = "Qwen/Qwen3-14b"

# Initialize clients
assistant_gen_client = GenerationServiceClient(base_url=f"http://localhost:5000")
eval_client = EvaluationServiceClient(base_url=f"http://localhost:5001")

In [2]:
# Load data and filter for LiveCodeBench tasks
dataset_fn = "data/sharded_instructions_600.json"
with open(dataset_fn, "r") as f:
    data = json.load(f)

data = [d for d in data if d["task"] == "code"]
livecodebench_tasks = [d for d in data if "livecodebench" in d["task_id"].lower()]

print(f"Found {len(livecodebench_tasks)} LiveCodeBench tasks")
print(f"Task IDs: {[d['task_id'] for d in livecodebench_tasks[:5]]}...")  # Show first 5


Found 55 LiveCodeBench tasks
Task IDs: ['sharded-livecodebench/2727', 'sharded-livecodebench/2728', 'sharded-livecodebench/2754', 'sharded-livecodebench/2755', 'sharded-livecodebench/2756']...


In [3]:
# Load model once
print(f"Loading model: {model_name}")
load_result = assistant_gen_client.load_model(model_name)
print(f"Model loaded: {load_result}")


Loading model: Qwen/Qwen3-14b
Model loaded: {'max_concurrent_jobs_per_worker': 15, 'max_context_length': 6000, 'message': 'Model Qwen/Qwen3-14b loaded successfully', 'num_gpus': 4, 'status': 'success', 'worker_status': {'gpus': {'gpu_0': {'active_jobs': 0, 'capacity_used': '0/15', 'gpu_busy': False, 'max_concurrent_jobs': 15, 'model_loaded': True, 'process_alive': True, 'process_pid': 1279006, 'utilization': '0/15'}, 'gpu_1': {'active_jobs': 0, 'capacity_used': '0/15', 'gpu_busy': False, 'max_concurrent_jobs': 15, 'model_loaded': True, 'process_alive': True, 'process_pid': 1279007, 'utilization': '0/15'}, 'gpu_2': {'active_jobs': 0, 'capacity_used': '0/15', 'gpu_busy': False, 'max_concurrent_jobs': 15, 'model_loaded': True, 'process_alive': True, 'process_pid': 1279008, 'utilization': '0/15'}, 'gpu_3': {'active_jobs': 0, 'capacity_used': '0/15', 'gpu_busy': False, 'max_concurrent_jobs': 15, 'model_loaded': True, 'process_alive': True, 'process_pid': 1279009, 'utilization': '0/15'}}}}


In [ ]:
# Evaluate model on each LiveCodeBench task
num_eval_runs = 100
results = []

for idx, sample in enumerate(livecodebench_tasks):
    task_id = sample["task_id"]
    print(f"[{idx+1}/{len(livecodebench_tasks)}] Evaluating task: {task_id}")
    
    task = get_task(sample["task"])
    system_message = task.generate_system_prompt(sample)
    input_prompt = task.populate_fully_specific_prompt(sample)
    conversation = [{"role": "system", "content": system_message}, {"role": "user", "content": input_prompt}]
    
    responses = generate_responses(assistant_gen_client, eval_client, sample, conversation, num_eval_runs)
    
    mean_score = np.mean([response["score"] for response in responses])
    print_colored(f"  Task {task_id}: Mean score = {mean_score:.4f}", "green")
    
    results.append({"task_id": task_id, "avg_performance": mean_score, "num_responses": len(responses)})

print("\nEvaluation complete!")


[1/55] Evaluating task: sharded-livecodebench/2727
  Task sharded-livecodebench/2727: Mean score = 0.2900
[2/55] Evaluating task: sharded-livecodebench/2728
  Task sharded-livecodebench/2728: Mean score = 0.0200
[3/55] Evaluating task: sharded-livecodebench/2754
  Task sharded-livecodebench/2754: Mean score = 0.2400
[4/55] Evaluating task: sharded-livecodebench/2755
  Task sharded-livecodebench/2755: Mean score = 0.0900
[5/55] Evaluating task: sharded-livecodebench/2756


In [ ]:
# Display results as pandas table
df = pd.DataFrame(results)
df = df.sort_values("avg_performance", ascending=False)
print(f"\nModel: {model_name}")
print(f"Number of tasks evaluated: {len(df)}")
print(f"Overall average performance: {df['avg_performance'].mean():.4f}")
print(f"\nResults:")
df
